# Python for Finance — Final practice session

In the final examination you will not have a computer: you will write your code by hand, on paper. 

Each exercise describes a business problem, gives its data and states the rules that your program must follow. All four exercises can be solved with the Python, NumPy and pandas seen in Sessions 1 and 2. Allow about **20 minutes** per exercise. Write it on a piece of paper first, then copy and run it here.

## Exercise 1 — Paying creditors in order of priority

**Context**

A small company has closed down. After selling everything it owned, the liquidator holds EUR 250k in cash, which must now be distributed among the company's creditors.

For each creditor, we know:

- the creditor's name;
- the creditor's class: `"staff"`, `"public"`, `"bank"` or `"supplier"`;
- the amount the company owes to that creditor, in k EUR.

The information is stored in three lists. The first entry in each list belongs to the first creditor, the second entry belongs to the second creditor, and so on. Each creditor appears once. The list `class_order` gives the four classes in their order of priority.

The cash is distributed according to the following rules:

1. The classes are served one after the other, in the order of `class_order`.
2. If the cash still available is at least the total amount owed to a class, every creditor of that class is paid in full.
3. Otherwise, the cash still available is shared within that class in proportion to the amounts owed: every creditor of the class receives the same fraction of what it is owed, where

   ```text
   fraction = cash still available / total amount owed to the class
   ```

   The cash is then exhausted, and the creditors of the following classes receive nothing.

**Final objective**

By the end of the exercise, your program should produce:

- `paid`: a dictionary that maps every creditor to the amount it receives, in thousand EUR (`0.0` for a creditor who receives nothing);
- `recovery`: a dictionary that maps every class to the fraction of the total amount owed to that class that is actually paid (`1.0` for a class paid in full, `0.0` for a class that receives nothing).

In [ ]:
creditors = ["Bank North", "Rossi", "Tax office", "Paper Co", "Bank South",
             "Dubois", "Ink Ltd", "Social security"]
classes = ["bank", "staff", "public", "supplier", "bank",
           "staff", "supplier", "public"]
owed = [150.0, 20.0, 60.0, 90.0, 50.0, 30.0, 60.0, 40.0]

class_order = ["staff", "public", "bank", "supplier"]
cash = 250.0

## Exercise 2 — Counting the votes of a shareholders' meeting

**Context**

The shareholders of a company vote on one resolution. Each shareholder has one vote per share, and the dictionary `register` gives the number of shares held by each shareholder.

Votes are sent electronically. The list `ballots` contains the ballots in the order in which they were received; each ballot is a tuple `(sender, choice)`.

The votes are counted according to the following rules:

1. A ballot is valid only if its sender appears in `register` and its choice is `"for"`, `"against"` or `"abstain"`. Invalid ballots are ignored.
2. A shareholder may send several ballots: only the shareholder's **last valid** ballot counts.
3. Quorum: the shareholders whose ballot counts, abstentions included, must together hold at least half of all the shares in `register`. Otherwise, the meeting cannot decide, and the outcome is `"no quorum"`.
4. If there is a quorum, the resolution is `"adopted"` if the shares voting `"for"` are more than half of the votes cast, and `"rejected"` otherwise. The votes cast are the shares voting `"for"` plus the shares voting `"against"`: abstentions are not votes cast.

**Final objective**

By the end of the exercise, your program should produce:

- `final_choice`: a dictionary that maps each shareholder whose ballot counts to the choice of that ballot;
- `shares_by_choice`: a dictionary that maps each of the three choices to the number of shares behind it (`0` if nobody made that choice);
- `outcome`: `"no quorum"`, `"adopted"` or `"rejected"`.

In [ ]:
register = {"Alba": 200, "Bruno": 75, "Chloe": 200,
            "Dev": 300, "Emma": 150, "Farid": 75}

ballots = [("Emma", "against"), ("Bruno", "for"), ("Gina", "for"),
           ("Chloe", "abstain"), ("Farid", "against"), ("Emma", "for"),
           ("Bruno", "yes")]

## Exercise 3 — Protecting positions with a trailing stop

**Context**

A fund buys three stocks, A, B and C, at their closing prices on day 0, and protects each position with a **trailing stop**. A trailing stop follows the highest closing price that the stock has reached since the purchase: if the price falls far enough below that level, the position is sold.

The array `prices` contains the closing prices over seven days: one row per day (day 0 to day 6) and one column per stock (A, B, C). The array `shares` gives the number of shares bought of each stock, in the same order as the columns.

The rules are:

1. A stock is sold at the closing price of the (first) day on which that price is at most 75% of the highest closing price of the stock from day 0 up to that day (both included).
2. Once a stock is sold, its later prices no longer matter.
3. A stock that is never sold is valued at its closing price on day 6.

The **exit price** of a stock is its sale price, or its day-6 price if it is never sold. The profit or loss of a position is

```text
profit or loss = number of shares × (exit price − closing price on day 0)
```

**Final objective**

By the end of the exercise, your program should produce:

- `exit_day`: a list with, for each stock in column order, the day on which it is sold, or `None` if it is never sold;
- `exit_price`: a NumPy array of shape `(3,)` with the exit price of each stock;
- `pnl`: a NumPy array of shape `(3,)` with the profit or loss of each position;
- `total_pnl`: the profit or loss of the three positions together.

The input arrays `prices` and `shares` must remain unchanged.

In [ ]:
import numpy as np

# closing prices: one row per day (day 0 to day 6), one column per stock (A, B, C)
prices = np.array([[ 80, 40, 120],
                   [ 84, 36, 110],
                   [ 96, 31, 125],
                   [ 88, 29, 118],
                   [ 72, 33, 100],
                   [ 70, 35,  95],
                   [ 90, 38, 108]])
shares = np.array([50, 100, 20])      # shares bought of A, B, C

## Exercise 4 — Reconciling positions with the custodian

**Context**

Every evening, the back office of an investment fund checks that the number of shares the fund believes it holds in each stock agrees with the custodian, the bank that keeps the shares on the fund's behalf.

The fund does not store its positions: they are rebuilt from the trade blotter `trades`, which has one row per trade, with:

- the trade ID;
- the ticker of the stock;
- the side, `"buy"` or `"sell"`;
- the number of shares traded;
- the status of the trade, `"settled"` or `"cancelled"`.

The custodian keeps the fund's shares in two accounts. Its report, `custodian_report`, has one row per account and stock, with the number of shares held.

The reconciliation follows these rules:

1. A cancelled trade never took place: it is ignored.
2. The fund position in a stock is the number of shares bought minus the number of shares sold in that stock.
3. The custodian position in a stock is the number of shares held in all of the fund's accounts together.
4. A stock that is missing on one side has a position of 0 on that side: a stock with no trade that took place has a fund position of 0, and a stock that does not appear in `custodian_report` has a custodian position of 0.
5. A stock is a **break** when its custodian position differs from its fund position.

**Final objective**

By the end of the exercise, your program should produce:

- `fund_position`: a Series, indexed by ticker, with the fund position in every stock that appears in at least one trade that took place;
- `difference`: a Series, indexed by ticker, with the custodian position minus the fund position, for every stock that appears in `fund_position` or in `custodian_report`;
- `breaks`: the part of `difference` that concerns the stocks that are breaks.

In [ ]:
import numpy as np
import pandas as pd

trades = pd.DataFrame({
    "trade_id": ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9"],
    "ticker": ["AAA", "BBB", "AAA", "CCC", "DDD", "BBB", "DDD", "CCC", "AAA"],
    "side": ["buy", "buy", "sell", "buy", "buy", "sell", "sell", "buy", "buy"],
    "quantity": [100, 50, 30, 80, 40, 20, 40, 60, 10],
    "status": ["settled", "settled", "settled", "cancelled", "settled",
               "settled", "settled", "settled", "cancelled"],
})

custodian_report = pd.DataFrame({
    "account": ["A-01", "A-01", "A-01", "A-02", "A-02"],
    "ticker": ["AAA", "BBB", "CCC", "AAA", "EEE"],
    "quantity": [50, 50, 60, 20, 25],
})